# Uber Fare Prediction Model

**Business objective:** estimate `fare_amount` from information available before or at trip start using the instructor-provided 50,000-row educational dataset.

> The supplied dataset does not contain `passenger_count`; this project does not fabricate it. Pickup/drop-off coordinates are retained only for data-quality review and are excluded from active prediction because they are not internally consistent with the supplied `distance_km`.

## Verified modeling workflow

The current workflow creates one 80/20 train/test split, performs **5-fold cross-validation only on the training partition**, selects the model with the lowest mean CV RMSE, fits that model on the full training partition, and evaluates it **once on the untouched holdout test set**. The holdout test scores are not used for model selection.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd() if (Path.cwd() / 'train.py').exists() else Path.cwd().parent
RAW_PATH = BASE / 'dataset/uber_trips_dataset_50k.csv'
CLEAN_PATH = BASE / 'dataset/uber_trips_dataset_50k_cleaned.csv'
TRAIN_READY_PATH = BASE / 'dataset/uber_trips_completed_training_ready.csv'

raw = pd.read_csv(RAW_PATH)
clean = pd.read_csv(CLEAN_PATH)
train_ready = pd.read_csv(TRAIN_READY_PATH)
print('raw:', raw.shape, 'clean:', clean.shape, 'training-ready:', train_ready.shape)

## Data-quality checks

In [ ]:
print('Missing values in raw:', int(raw.isna().sum().sum()))
print('Duplicate raw rows:', int(raw.duplicated().sum()))
print('Duplicate trip IDs:', int(raw['trip_id'].duplicated().sum()))
print('Completed training rows:', len(train_ready))
print('Passenger count present:', 'passenger_count' in raw.columns)

## Core business analysis

In [ ]:
eda = clean.copy()
eda['pickup_time'] = pd.to_datetime(eda['pickup_time'], errors='coerce')
eda['pickup_hour'] = eda['pickup_time'].dt.hour
eda['pickup_month'] = eda['pickup_time'].dt.month
eda['day_name'] = eda['pickup_time'].dt.day_name()

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(eda['fare_amount'].dropna(), bins=30)
ax.set(title='Fare distribution', xlabel='Fare', ylabel='Trips')
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
sample = eda.dropna(subset=['distance_km','fare_amount']).sample(min(5000, len(eda)), random_state=42)
ax.scatter(sample['distance_km'], sample['fare_amount'], alpha=.35, s=10)
ax.set(title='Fare vs supplied distance', xlabel='Distance (km)', ylabel='Fare')
plt.show()

## Average fare by day of week

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_avg = eda.groupby('day_name', as_index=False)['fare_amount'].mean()
day_avg['day_name'] = pd.Categorical(day_avg['day_name'], categories=day_order, ordered=True)
day_avg = day_avg.sort_values('day_name')
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(day_avg['day_name'].astype(str), day_avg['fare_amount'])
ax.set(xlabel='Day of week', ylabel='Average fare', title='Average fare by day of week')
ax.tick_params(axis='x', rotation=30)
plt.show()
day_avg

## Reproduce model selection and final evaluation

In [ ]:
from train import run

# Use a new/empty folder for each run.
# Example:
# metadata = run(BASE / 'work/run_notebook_001')
# metadata

## Interpretation

`train.py` is the authoritative training implementation. It compares Linear Regression, Random Forest, and Gradient Boosting using 5-fold KFold cross-validation on the training partition only, selects the lowest mean CV RMSE, then evaluates the selected model once on the untouched holdout set.

The Streamlit app provides the interactive fare estimate, average fare by day of week, feature influence/coefficient visualization, and Actual vs Predicted diagnostic.